In [145]:
import os,sys
import json
import pandas as pd

from rockyclickup.wrapper import Session as cu_sesh
from rockyclickup.database_interface import get_all_fields, get_all_tasks
from rockyclickup.models import MODEL_LOOKUP
from rockyclickup.utils import response_to_dataframe as rcu_response_to_dataframe

from rockyelevate.wrapper import Session
from rockyelevate.utils import response_to_dataframe

current_dir = os.getcwd()
parent_dir = os.path.dirname(f"{"\\".join(current_dir.split("\\")[:-1])}")
sys.path.append(parent_dir)

from constants.maps import ELV_STATUS_MAP, ELV_ACCOUNT_TYPE_MAP
from utils.collector import get_all_cu_plans, get_all_cu_clients, get_all_elv_organizations, get_all_elv_plans

In [40]:
elv = Session("PROD", multithread=True, max_threads=40)
clickup = cu_sesh()

In [ ]:
new_plan_data = pd.read_csv("3_after_update_naked_bodies.csv")

In [ ]:
cu_clients = get_all_cu_clients()


In [93]:
elv_orgs = get_all_elv_organizations()
elv_plans = get_all_elv_plans(oids=[int(i) for i in elv_orgs['organization_id'].to_list()])
cu_clients = get_all_cu_clients()
cu_plans = get_all_cu_plans()

opened 1594 organizations
opened 5918 elevate plans
fetching clients from clickup, please wait...
Field not in config.db: update_account_managers 702d85f6-7155-447c-9019-206db17ab27c
Field not in config.db: ducks 80f59460-7b7d-4652-9d9c-f1029b146ede
Field not in config.db: data_transmission_details 2bd919bf-6695-4f13-9f80-9e7b077bc83c
Field not in config.db: divisional_invoicing 951e9aa5-c6d9-4ad1-98a9-118b5d478640
Field not in config.db: temp_am_cobra f1c08a6f-3a1c-4392-8967-bbcd9501e1a8
Field not in config.db: temp_account_manager 159d31bb-e617-4d3a-b900-021cc0ad4542
Field not in config.db: data_start 7d44c3ea-fbe2-42d7-a21a-0e8f42bd8f1b
Field not in config.db: temp_flex_divisions 0d2211c9-b8c9-417e-b020-74ca30273509
Field not in config.db: temp_cobra_divisions e1798c36-372a-446b-8612-de839429d43c
Field not in config.db: data_end 3b8c9ecd-1e23-442a-8155-2b15c1559dee
found 2735 clickup clients
opened 6053 clickup plans


In [ ]:
new_plan_df = elv_plans[elv_plans['elv_plan_id'].isin(new_plan_data['new_elv_plan_id'])]

for col in ['plan_year.valid_from', 'plan_year.valid_to']:
    new_plan_df[col] = pd.to_datetime(new_plan_df[col])

In [ ]:
print([c for c in cu_clients if "admin" in c])

In [ ]:
rmrcode_map = {
    r.get("organization_id"): r.get("rmrcode") for _,r in elv_orgs.iterrows()
}

am_map = {
    r.get("organization_id"): r.get("account_manager") for _, r in elv_orgs.iterrows()
}

date_admin_start = {
    r.get("rmrcode"): r.get("cu_client_date_admin_start") for _, r in cu_clients.iterrows()
}

new_plan_df['rmrcode'] = new_plan_df['organization_id'].map(rmrcode_map)
new_plan_df['cu_plan_account_manager'] = new_plan_df['organization_id'].map(am_map)
new_plan_df['cu_client_date_admin_start'] = new_plan_df['rmrcode'].map(date_admin_start)

In [ ]:
print([c for c in new_plan_df.columns if "cu_acc" in c])

In [56]:
''' generate rcu models to post to clickup '''

all_db_custom_fields = get_all_fields()

db_field_map = {
    db_f.custom_name: db_f for db_f in all_db_custom_fields
}

db_tasks = get_all_tasks()

list_id_map = {
    t.name.upper(): t.list_id for t in db_tasks
}

def generate_rcu_model(row):

    cu_account_type = ELV_ACCOUNT_TYPE_MAP.get(row['account_type.account_type'])

    try:
        list_id = list_id_map.get(cu_account_type)
        rcu_model = MODEL_LOOKUP.get(list_id)
    except Exception as e:
        print(e)
        raise

    detail_dict = {
        "list_id": list_id
    }

    # new_elv_plan = response_to_dataframe([row['new_plan_res']]).iloc[0]

    ''' task name '''
    plan_name = f'{row['rmrcode'].upper()} {cu_account_type.upper()}'
    if cu_account_type.upper() != 'HSA' and row['account_type.account_type'].upper() != 'HSA':
        plan_name = f'{plan_name} {row['plan_year.valid_from'].year}'
    detail_dict['name'] = plan_name

    ''' status '''
    detail_dict['status'] = ELV_STATUS_MAP.get(row['elv_plan_status'], 'review')

    ''' account manager '''
    detail_dict['account_manager'] = row['cu_account_manager'] or row['cu_account_manager']

    ''' date plan start '''
    detail_dict['date_plan_start'] = row['plan_year.valid_from']

    ''' date plan end '''
    detail_dict['date_plan_end'] = row['plan_year.valid_to']

    ''' elevate id '''
    detail_dict['elv_id'] = row['elv_plan_id']

    ''' plan code '''
    detail_dict['elv_plan_code'] = row['plan_code']

    ''' date admin start '''
    detail_dict['date_admin_start'] = row['cu_client_date_admin_start']

    ''' card '''
    detail_dict['card'] = row['plan_primary_config.is_carded.is_carded']

    ''' annual election max '''
    if row['plan_primary_config.max_election_amount_type.max_election_amount_type'] == 'IRS_LIMIT':
        
        detail_dict['annual_election_max'] = row['plan_primary_config.max_election_amount_type.max_election_amount'] / 2
        detail_dict['annual_election_auto_adjust'] = True
    else:
        detail_dict['annual_election_max'] = row['plan_primary_config.max_election_amount_type.max_election_amount']
        detail_dict['annual_election_auto_adjust'] = False

    ''' annual election min '''
    detail_dict['annual_election_min'] = row['plan_primary_config.min_election_amount_type.min_election_amount']

    ''' run out '''
    detail_dict['run_out'] = row['plan_coverage_config.run_out_type.run_out_days_amount']

    ''' run out termed ee '''
    detail_dict['run_out_termed_ee'] = row['plan_coverage_config.claims_deadline_end_of_coverage_type.claims_deadline_end_of_coverage_days_amount']

    ''' run out termed ee matches plan '''
    if row['plan_coverage_config.claims_deadline_end_of_coverage_type.claims_deadline_end_of_coverage_type'] == 'PRE_DEFINED_END_DATE':
        detail_dict['run_out_termed_ee_matches_plan'] = True
    else: 
        detail_dict['run_out_termed_ee_matches_plan'] = False
        
    ''' grace period '''
    detail_dict['grace_period'] = row['plan_coverage_config.grace_period_type.grace_period_days_amount']

    ''' rollover '''
    detail_dict['rollover'] = row['plan_account_funding_config.is_rollover.is_rollover']
    
    ''' rollover max '''
    detail_dict['rollover_max'] = row['plan_account_funding_config.max_rollover_amount.max_rollover_amount']
    
    ''' rollover min '''
    detail_dict['rollover_min'] = row['plan_account_funding_config.min_rollover_amount.min_rollover_amount']

    ''' lpf '''
    detail_dict['lpf'] = row.get('lpf', False)

    # remove keys not found on model
    keys_not_in_model = [k for k in detail_dict.keys() if k not in dir(rcu_model)]
    for key in keys_not_in_model:
        detail_dict.pop(key)

    # remove na values
    na_key_values = [k for k, v in detail_dict.items() if isinstance(v, list) or pd.isna(v)]
    for key in na_key_values:
        detail_dict[key] = None

    # remove formula fields
    formula_fields = [k for k in detail_dict.keys() if 'formula' in k]
    for key in formula_fields:
        detail_dict.pop(key)

    # enforce types
    for key, value in detail_dict.items():
        db_field = db_field_map.get(key, None)
        if not db_field:
            types_not_enforced.append(key)
            continue
        
        if pd.isna(value):
            continue

        field_type = db_field.type

        if field_type in ['number']:
            detail_dict[key] = int(value)
            continue

        if field_type in ['currency']:
            detail_dict[key] = float(value)
            continue

        if field_type in ['checkbox']:
            detail_dict[key] = bool(value)
            continue

        if field_type in ['text', 'short_text']:
            detail_dict[key] = str(value)

    # 'automatic_progress',
    # 'date',
    # 'drop_down',
    # 'email',
    # 'formula',
    # 'labels',
    # 'list_relationship',
    # 'location',
    # 'manual_progress',
    # 'phone',
    # 'url',
    # 'users'

        types_not_enforced.append(field_type)

        if field_type in ['phone']:
            print(f'phone value: {value}')

    missing_keys.extend([k for k in dir(rcu_model) if k in db_field_map and k not in detail_dict])
    populated_model = rcu_model(**detail_dict)
    return populated_model


missing_keys = []
types_not_enforced = []

rcu_models = new_plan_df.apply(generate_rcu_model, axis=1).to_list()


In [ ]:
rcu_models

In [57]:
print(rcu_model.list_id)

None


In [59]:
clickup_responses = []

for rcu_model in rcu_models:
    clickup_res = clickup.create(rcu_model)
    clickup_responses.append(clickup_res)

In [94]:
account_manager_map = {
    r.get("client_id"): r.get("cu_client_account_manager") for _, r in cu_clients.iterrows()
}

client_id_map = {
    r.get("rmrcode"): r.get("client_id") for _, r in cu_clients.iterrows()
}

In [95]:
cu_clients[cu_clients['rmrcode'] == 'RMRDNS']

,client_id,client_name,cu_client_description,cu_client_archived,cu_client_assignees,cu_client_assignees,cu_client_tags,cu_client_url,cu_client_status,status.id,...,ducks,data_transmission_details,divisional_invoicing,temp_am_cobra,temp_account_manager,data_start,temp_flex_divisions,temp_cobra_divisions,data_end,year_oe_approved
2336,86877zkum,Denovo Solutions,,False,[],[],[],https://app.clickup.com/t/86877zkum,open enrollment,sc901102596133_rcjNc01P,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [146]:
creation_responses = rcu_response_to_dataframe(clickup_responses)

Field not in config.db: structure 5515a85d-d6fb-45bf-8e40-081e41b5e11b


In [148]:
creation_responses.to_csv("4_creation_on_clickup.csv")

In [ ]:
db_fields = get_all_fields()


relation_fields = {
    f.custom_name.split("_")[1].upper(): f.field_id
    for f in db_fields
    if "client_" in f.custom_name
    and len(f.custom_name) == 10
}

am_field_id = [f for f in db_fields if "account_manager" == f.custom_name][0].field_id

print(am_field_id)


8abc6fab-68c6-4f52-94a8-2767861b070c


In [137]:
link_client_responses = []
link_am_responses = []



for res in clickup_responses:
    # print(res)
    task_id = res.get('id')

    rmrcode = res.get("name").split()[0].strip()

    client_id = client_id_map.get(rmrcode)

    account_managers = account_manager_map.get(client_id)

    list_id = res.get('list').get('id')

    relation_field_id = relation_fields.get(res.get('list').get('name'))

    link_client_res = clickup.patch(
        task_id=task_id,
        field_id=relation_field_id,
        add=[client_id]
    )

    link_client_responses.append(link_client_res)

    link_am_res = clickup.patch(
        task_id=task_id,
        field_id=am_field_id,
        add=account_managers
    )

    # link_am_responses.append(link_am_res)


In [ ]:

final_df = new_plan_data.copy()
missing_rows = plan_df[~plan_df.index.isin(final_df.index)]

print("final_df columns:", final_df.columns.tolist())
print("Duplicate columns in final_df:", final_df.columns[final_df.columns.duplicated()].tolist())
print("missing_rows columns:", missing_rows.columns.tolist())
print("Duplicate columns in missing_rows:", missing_rows.columns[missing_rows.columns.duplicated()].tolist())

final_df = final_df.loc[:, ~final_df.columns.duplicated()]
missing_rows = missing_rows.loc[:, ~missing_rows.columns.duplicated()]
missing_rows_reindexed = missing_rows.reindex(columns=final_df.columns)

after_df = pd.concat([final_df, missing_rows], ignore_index=True)

after_df = after_df[[c for c in after_df.columns if c != 'error'] + ['error']]

after_df.to_pickle("SEP_OCT_ROLLED_FINAL.pkl")
